In [ ]:
import json
import pandas as pd

from datetime import datetime, timezone, date

# Add Questions here
questions = [
    {"id": 1, "text": "What is your name?", "type": "text"},
    {"id": 6, "text": "What is your date of birth? (YYYY-MM-DD)", "type": "dob"},
    {"id": 3, "text": "What is your gender?", "choices": ["Male", "Female", "Other"]},
    {"id": 4, "text": "What is your height? (centimeters)", "type": "number"},
    {"id": 5, "text": "What is your weight? (lbs)", "type": "number"},
    {"id": 2, "text": "What is your race or ethnicity", "choices": ["Caucasian", "AfricanAmerican", "Hispanic", "Asian", "Other"]},
    {"id": 7, "text": "How many times were you in the hospital for this specific issue?", "type": "number"},
    {"id": 8, "text": "How many different lab procedures were you tested for?", "type": "number"},
    {"id": 9, "text": "How many different medications have you been using for this issue?", "type": "number"},
    {"id": 10, "text": "How many times have you been to the emergency room for this specific issue?", "type": "number"},
    {"id": 11, "text": "How many times have you been to the hospitalized for this specific issue?", "type": "number"},
    {"id": 12, "text": "How many times have you been diagnosed for this specific issue?", "type": "number"},
    {"id": 13, "text": "What was your blood glucose test?", "choices": ["NoTest", "Norm", ">200", ">300"]},
    {"id": 14, "text": "What was your A1C blood sugar test?", "choices": ["NoTest", "Norm", ">7", ">8"]},
    {"id": 15, "text": "Are you currently taking any medications for diabetes?", "choices": ["No", "Yes"]},
    {"id": 16, "text": "On your last hospital visit have your diabetes medications changed?", "choices": ["No", "Changed"]},
    {"id": 17, "text": "On your last hospital visit has your metaformin dosage changed if you ever had one?", "choices": ["No", "Down", "Steady", "Up"]},
    {"id": 18, "text": "On your last hospital visit has your insulin dosage changed if you ever had one?", "choices": ["No", "Down", "Steady", "Up"]},
]

# Define Questionaire
def run_questionnaire(q_list):
    answers = {}  # {question_id: selected_option_index}

    print("Questionnaire")
    print("----------------------------")

    for q in q_list:
        print(f"\n{q['text']}", flush=True)
        
        # DOB question
        if q.get("type") == "dob":
            while True:
                user_in = input("Enter DOB (YYYY-MM-DD): ").strip()
                try:
                    birth_date = datetime.strptime(user_in, "%Y-%m-%d").date()
                    answers[q["id"]] = birth_date.isoformat()
                    break
                except ValueError:
                    print("Invalid format. Use YYYY-MM-DD.")

        # number question
        elif q.get("type") == "number":
            while True:
                user_in = input("Enter a number: ").strip()
                if user_in.isdigit():
                    answers[q["id"]] = int(user_in)
                    break
                else:
                    answers[q["id"]] = "?"
                    break
        
        # text question
        elif q.get("type") == "text":
            while True:
                user_in = input("Enter your answer: ").strip()
                if user_in:  # ensure not empty
                    answers[q["id"]] = user_in
                    break
                print("Please enter a valid response.")


        # multiple choice
        else:
            for idx, choice in enumerate(q["choices"], start=1):
                print(f"  {idx}) {choice}")

            while True:
                user_in = input(f"Select 1-{len(q['choices'])}: ").strip()
                if user_in.isdigit():
                    pick = int(user_in)
                    if 1 <= pick <= len(q["choices"]):
                        answers[q["id"]] = pick
                        break
                print("Please enter a valid option number.")

    return answers

def get_age_range(age):
    if 0 <= age < 10:
        return "[0-10)"
    elif 10 <= age < 20:
        return "[10-20)"
    elif 20 <= age < 30:
        return "[20-30)"
    elif 30 <= age < 40:
        return "[30-40)"
    elif 40 <= age < 50:
        return "[40-50)"
    elif 50 <= age < 60:
        return "[50-60)"
    elif 60 <= age < 70:
        return "[60-70)"
    elif 70 <= age < 80:
        return "[70-80)"
    elif 80 <= age < 90:
        return "[80-90)"
    elif 90 <= age < 100:
        return "[90-100)"

def calculate_age(birth_date):
    today = date.today()
    return today.year - birth_date.year - (
        (today.month, today.day) < (birth_date.month, birth_date.day)
    )

answers = run_questionnaire(questions)

# store session input
session = {
    "session_id": datetime.now(timezone.utc).isoformat(),
    "answers": answers
}

# readable summary
answer_summary = {}

for q in questions:
    qid = q["id"]
    answer = answers.get(qid)

    if answer is not None:

        # DOB → age → range
        if q.get("type") == "dob":
            birth_date = datetime.strptime(answer, "%Y-%m-%d").date()
            age = calculate_age(birth_date)
            age_range = get_age_range(age)

            answer_summary[q["text"]] = f"{age_range}"

        # number questions
        elif q.get("type") == "number":
            answer_summary[q["text"]] = answer
        # text questions
        elif q.get("type") == "text":
            answer_summary[q["text"]] = answer

        # multiple choice
        else:
            answer_summary[q["text"]] = q["choices"][answer - 1]

print("\n--- Answer Summary (in memory) ---")
for k, v in answer_summary.items():
    print(f"{k}: {v}")

id_to_text = {q["id"]: q["text"] for q in questions}
for qid in [1, 4, 5]:
    text_key = id_to_text.get(qid)
    if text_key:
        answer_summary.pop(text_key, None)

patients_data = list(answer_summary.values())
patients_data[:3] = patients_data[:3][::-1]


# save to json (if needed)
out_file = "session_answers.json"
to_save = {
    "session_id": session["session_id"],
    "questions": questions,
    "answers": answers,
    "answer_summary": answer_summary
}


with open(out_file, "w") as f:
    json.dump(to_save, f, indent=2)


print(patients_data)


print("\nSaved to:", out_file)

Questionnaire
----------------------------

What is your name?

What is your date of birth? (YYYY-MM-DD)

What is your gender?
  1) Male
  2) Female
  3) Other

What is your height? (meters)

What is your weight? (lbs)

What is your race or ethnicity
  1) Caucasian
  2) AfricanAmerican
  3) Hispanic
  4) Asian
  5) Other

How many times were you in the hospital for this specific issue?

How many different lab procedures were you tested for?

How many different medications have you been using for this issue?

How many times have you been to the emergency room for this specific issue?

How many times have you been to the hospitalized for this specific issue?

How many times have you been diagnosed for this specific issue?

What was your blood glucose test?
  1) NoTest
  2) Norm
  3) >200
  4) >300

What was your A1C blood sugar test?
  1) NoTest
  2) Norm
  3) >7
  4) >8

Are you currently taking any medications for diabetes?
  1) No
  2) Yes

On your last hospital visit have your diabet

In [32]:
RACE = 0
GENDER = 1
AGE = 2
A1C = 9
GLUCOSE = 10
DIABETES_MED = 11
CHANGE = 12
METFORMIN = 13
INSULIN = 14


if patients_data[RACE] == 'AfricanAmerican':
    patients_data[RACE] = 1
elif patients_data[RACE] == 'Caucasian':
    patients_data[RACE] = 0
elif patients_data[RACE] == 'Hispanic':
    patients_data[RACE] = 2
elif patients_data[RACE] == 'Asian':
    patients_data[RACE] = 3
else:
    patients_data[RACE] = 4

if patients_data[GENDER] == 'Male':
    patients_data[GENDER] = 0
elif patients_data[GENDER] == 'Female':
    patients_data[GENDER] = 1
else:
    patients_data[GENDER] = 2

age_list = [
    '[0-10)', '[10-20)', '[20-30)', '[30-40)',
    '[40-50)', '[50-60)', '[60-70)', '[70-80)',
    '[80-90)', '[90-100)'
]
patients_data[AGE] = age_list.index(patients_data[AGE])

if patients_data[A1C] == 'NoTest':
    patients_data[A1C] = 0
elif patients_data[A1C] == 'Norm':
    patients_data[A1C] = 1
elif patients_data[A1C] == '>200':
    patients_data[A1C] = 2
elif patients_data[A1C] == '>300':
    patients_data[A1C] = 3

if patients_data[GLUCOSE] == 'NoTest':
    patients_data[GLUCOSE] = 0
elif patients_data[GLUCOSE] == 'Norm':
    patients_data[GLUCOSE] = 1
elif patients_data[GLUCOSE] == '>7':
    patients_data[GLUCOSE] = 2
elif patients_data[GLUCOSE] == '>8':
    patients_data[GLUCOSE] = 3

def med_map(x):
    if x == 'No':
        return 0
    elif x == 'Down':
        return 1
    elif x == 'Steady':
        return 2
    elif x == 'Up':
        return 3

patients_data[METFORMIN] = med_map(patients_data[METFORMIN])
patients_data[INSULIN] = med_map(patients_data[INSULIN])

patients_data[DIABETES_MED] = 0 if patients_data[DIABETES_MED] == 'No' else 1
patients_data[CHANGE] = 0 if patients_data[CHANGE] == 'No' else 1





In [43]:
import pandas as pd
import joblib

# Load saved items
model = joblib.load("dfu_model.pkl")
scaler = joblib.load("scaler.pkl")
feature_order = joblib.load("feature_order.pkl")

# patients_data is a LIST (single patient)
# Convert list → DataFrame using feature_order as columns
input_df = pd.DataFrame([patients_data], columns=feature_order)

# Scale input
input_scaled = scaler.transform(input_df)

# Predict
prediction = model.predict(input_scaled)[0]

# Optional probability
prob = model.predict_proba(input_scaled)[0][1]

if prediction == 0:
    print(f"We predict you do not have a DFU with a risk of getting an ulcer at {prob*100:.2f}%")
else:
    print("We predict that you have a Diabetic Foot ulcer.")

We predict you do not have a DFU with a risk of getting an ulcer at 0.46%
